In [ ]:
from pathlib import Path
import sys

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root / "src"))

from walinet.parameter_calibration.load_data import *

from walinet.parameter_calibration.compute_statistics import (
    extract_valid_voxels,
    calculate_pooled_median_iqr,
)

from walinet.parameter_calibration.metab_calibration import *

from walinet.parameter_calibration.water_lipid_ratios import *

from walinet.parameter_calibration.pipeline_FWHM_SNR_shifts import (
    calibrate_parameter_from_maps,
)



In [ ]:
bandwidth_hz = 2778.0
nmr_frequency_hz = 297_222_931.0
water_ppm = 4.68

In [ ]:
SUBJECT_DIRS = [
    "/ceph/mri.meduniwien.ac.at/departments/radiology/mrsbrain/home/bstrasser/Projects/Project9_ImplementRecoInICE/Step5_MultiCenterStudy/LargeData_d3hj/Results/Brisbane/Vol03_Dat_NoL2_GradDel",
    "/ceph/mri.meduniwien.ac.at/departments/radiology/mrsbrain/home/bstrasser/Projects/Project9_ImplementRecoInICE/Step5_MultiCenterStudy/LargeData_d3hj/Results/Brisbane/Vol04_Dat_NoL2_GradDel",
    "/ceph/mri.meduniwien.ac.at/departments/radiology/mrsbrain/home/bstrasser/Projects/Project9_ImplementRecoInICE/Step5_MultiCenterStudy/LargeData_d3hj/Results/Brisbane/Vol05_Dat_NoL2_GradDel",
    "/ceph/mri.meduniwien.ac.at/departments/radiology/mrsbrain/home/bstrasser/Projects/Project9_ImplementRecoInICE/Step5_MultiCenterStudy/LargeData_d3hj/Results/Brisbane/Vol07_Dat_NoL2_GradDel",
    "/ceph/mri.meduniwien.ac.at/departments/radiology/mrsbrain/home/bstrasser/Projects/Project9_ImplementRecoInICE/Step5_MultiCenterStudy/LargeData_d3hj/Results/London/Vol01_Dat_NoL2_GradDel",
    "/ceph/mri.meduniwien.ac.at/departments/radiology/mrsbrain/home/bstrasser/Projects/Project9_ImplementRecoInICE/Step5_MultiCenterStudy/LargeData_d3hj/Results/London/Vol02_Dat_NoL2_GradDel",
    "/ceph/mri.meduniwien.ac.at/departments/radiology/mrsbrain/home/bstrasser/Projects/Project9_ImplementRecoInICE/Step5_MultiCenterStudy/LargeData_d3hj/Results/London/Vol03_Dat_NoL2_GradDel",
    "/ceph/mri.meduniwien.ac.at/departments/radiology/mrsbrain/home/bstrasser/Projects/Project9_ImplementRecoInICE/Step5_MultiCenterStudy/LargeData_d3hj/Results/London/Vol04_Dat_NoL2_GradDel",
    "/ceph/mri.meduniwien.ac.at/departments/radiology/mrsbrain/home/bstrasser/Projects/Project9_ImplementRecoInICE/Step5_MultiCenterStudy/LargeData_d3hj/Results/London/Vol05_Dat_NoL2_GradDel"
]

In [ ]:
metabolite_reference = load_metabolite_reference_data(
    base_paths=SUBJECT_DIRS,
    mask_relative_path="maps/mask",
    fit_relative_path="maps/SpecMap_LCMFit",
    baseline_relative_path="maps/SpecMap_LCMBaseline",
    extension=".nii.gz",
)

In [ ]:
Metabos = ["Asp", "Cr", "GABA", "Glc", "Gln", "Glu", "GPC", "GSH", "Ins", "NAA", "NAAG", "PCh", "PCr", "Scyllo", "Tau", "TwoHG"]

metabolite_results = {}

for Metab in Metabos:
    metabolite_results[Metab] = (
        calibrate_metabolite_ratio_from_map(
            reference_data=metabolite_reference,
            base_paths=SUBJECT_DIRS,
            relative_path=f"maps/Orig/{Metab}_amp_map",
            extension=".nii.gz",
            metabolite_name=Metab,
            crlb_threshold=None,
            bins=30,
            plot_percentile=99.5,
            lognormal_sigma_factor=2,
        )
    )